## Adaptive Workflow Pattern

From the article on [Top AI Agentic Workflow Patterns (2026)](https://lekha-bhan88.medium.com/top-ai-agentic-workflow-patterns-that-will-shape-ai-systems-in-2026-736a3141d0e0).

The Adaptive Workflow pattern is the most architecturally complex of the six patterns. The core idea: instead of a fixed pipeline where every task goes through the same sequence of agents, an orchestrator LLM decides at runtime which specialized sub-agents to invoke, in what order, and whether to loop back or terminate — based on the specific input and the results it receives along the way.

Every other pattern in this set has a static structure. Reflection always generates-critiques-refines. Planning always plans-then-executes. Even multi-agent systems typically have fixed routing between agents. Adaptive Workflow is the one where the execution graph itself is dynamic — the path through the system is determined by the LLM at each decision point, not hardcoded by the developer.

The diagram from the article shows the key elements:
- An **Orchestrator** at the top that receives the input and decides what to do
- A pool of **specialized sub-agents** (Agent 1, Agent 2, Agent n)
- The orchestrator can call agents in sequence or in parallel, pass outputs between them, and loop back for additional processing if needed
- A **synthesis step** that assembles the final output from sub-agent results

The result is a system that can handle a wide variety of inputs with a single entry point, routing each request to the right combination of capabilities.

### What makes it adaptive

The word "adaptive" refers to two distinct things in this pattern:

**Adaptive routing** — the orchestrator selects which sub-agents to invoke based on the input. A simple question might go directly to a single research agent. A complex multi-part question might fan out across three agents in parallel, then get synthesized. The workflow topology changes per request.

**Adaptive iteration** — the orchestrator can decide to loop. If a sub-agent returns low-confidence output, or if a validation step fails, the orchestrator can re-route to a different agent, request a revised answer, or escalate to a human — rather than always terminating after a fixed number of steps.

This is the key difference from the Planning pattern. A planner produces a complete plan upfront and executes it linearly. An adaptive workflow decides the next step *after* seeing each result. It can react to what it finds.

And it's different from a fixed multi-agent setup. In a fixed setup, every request goes through Agent A → Agent B → Agent C. In adaptive workflow, some requests go A → C, some go B → A → C, some go A and B in parallel → C. The orchestrator makes this decision at runtime.

### Anatomy of an Adaptive Workflow

A minimal implementation has four components:

**1. Orchestrator** — an LLM that receives the user request, understands what capabilities are available, and decides the execution plan. It outputs structured routing decisions (which agents to call, with what inputs) rather than content.

**2. Agent registry** — a catalogue of available sub-agents with their capabilities and input/output contracts. The orchestrator uses this to make routing decisions. If the orchestrator doesn't know what agents exist, it can't decide which to call.

**3. Sub-agents** — specialized units with a single well-defined capability (e.g., web research, code execution, data analysis, summarization). Each agent does one thing well. Specialization is what makes the routing decision meaningful — there's no point in having a router if all agents do the same thing.

**4. Synthesizer** — assembles the final response from the outputs of all invoked sub-agents. This can be as simple as concatenation, or another LLM call that integrates the outputs into a coherent answer.

```
User Request
     │
     ▼
Orchestrator (LLM)
     │
     ├── Decision: invoke Agent A and Agent B in parallel
     │        │
     │        ├── Agent A: research recent developments
     │        └── Agent B: retrieve historical context
     │
     ├── Receive outputs from A and B
     │
     ├── Decision: quality check passed → proceed to synthesis
     │   (or: quality check failed → loop Agent A with refined query)
     │
     ▼
Synthesizer (LLM)
     │
     ▼
Final Response
```

### Use Cases

Adaptive workflow is appropriate when:
- The system needs to handle a wide variety of request types with a single entry point
- Different requests genuinely require different combinations of capabilities
- Some requests need iteration (retry, refine, escalate) while others can complete in one pass
- The cost of running all agents on every request would be prohibitive

**Customer service systems** — a single agent receives all inbound requests. Billing queries go to the billing agent. Technical issues go to the technical support agent. Complex issues requiring both account history and technical context get both. Returns require the order agent and the policy agent. All of this is decided at runtime based on the specific message.

**Research and analysis platforms** — a user asks "Give me a competitive analysis of electric vehicle charging infrastructure in Europe." The orchestrator might route to: (1) a data retrieval agent for market statistics, (2) a news research agent for recent developments, (3) a company analysis agent for key players. The outputs are synthesized into a single report. A simpler question like "What is the range of the latest Tesla Model S?" might just go to a single product research agent.

**Coding assistants** — a request to "fix this bug" goes to the debugging agent. A request to "add a feature and write tests" goes to the implementation agent and the test-writing agent in parallel. A request to "review this for security issues" goes to the security review agent specifically. Fixed pipelines would either under-provision (simple questions go through unnecessary agents) or under-serve (complex questions lack needed capabilities).

**Document processing** — route invoices to the extraction agent, contracts to the legal analysis agent, technical specs to the structured data agent. The orchestrator classifies the document type and routes accordingly, rather than running all three parsers on every document.

### Benefits and Limitations

**Benefits:**

**Efficiency** — only the agents relevant to the specific request are invoked. A simple factual question doesn't need to run through a five-agent pipeline. This has real cost implications at scale.

**Flexibility** — the system handles edge cases gracefully. When a novel request type arrives that doesn't fit the standard routing path, the orchestrator can improvise a combination of existing agents rather than failing entirely.

**Maintainability** — adding a new capability means adding a new sub-agent to the registry. You don't need to rewire the routing logic in code — you just update the orchestrator's description of available capabilities. The orchestrator figures out when to use the new agent.

**Quality control** — the orchestrator can include a validation step after sub-agent outputs. If a result doesn't meet quality criteria, it can retry with a different agent or a refined prompt before synthesizing.

**Limitations:**

**Orchestrator reliability** — the entire system depends on the orchestrator making good routing decisions. If it miscategorizes a request or picks the wrong agent, all downstream work is affected. Orchestrator prompt engineering is the critical failure point.

**Debugging complexity** — because the execution path is dynamic, reproducing a specific failure requires the same input, the same orchestrator decision, and the same sub-agent outputs. Logging every routing decision is essential.

**Latency unpredictability** — fixed pipelines have predictable latency. Adaptive workflows don't — a complex request might invoke five agents sequentially, while a simple one finishes in one step. This can make SLA planning difficult.

**Cost unpredictability** — the same issue applies to cost. Simple requests are cheap; complex ones are expensive. Budget estimates require understanding the distribution of request complexity, not just the average.

### Implementation Example

In [ ]:
from dataclasses import dataclass
from typing import Any


@dataclass
class AgentDefinition:
    agent_id: str
    name: str
    description: str  # shown to orchestrator for routing decisions
    handler: Any  # callable that executes the agent


# --- Sub-agents ---

def research_agent(query: str) -> str:
    """Finds factual information from external sources."""
    print(f"  [ResearchAgent] Researching: {query}")
    return f"Research findings for '{query}': [simulated search results with key facts]"


def analysis_agent(data: str) -> str:
    """Analyzes data and extracts insights."""
    print(f"  [AnalysisAgent] Analyzing: {data[:60]}...")
    return f"Analysis complete: key trends identified, 3 main insights extracted from the data."


def summarization_agent(content: str) -> str:
    """Produces a concise summary of longer content."""
    print(f"  [SummarizationAgent] Summarizing: {content[:60]}...")
    return f"Summary: concise 3-sentence summary of the provided content."


def code_agent(task: str) -> str:
    """Generates or reviews code for a given task."""
    print(f"  [CodeAgent] Handling: {task}")
    return f"Code output for '{task}': [generated Python code snippet]"


# --- Agent Registry ---

AGENT_REGISTRY: list[AgentDefinition] = [
    AgentDefinition(
        agent_id="research",
        name="Research Agent",
        description="Finds factual information, data, and current events from external sources. Best for questions requiring up-to-date information.",
        handler=research_agent,
    ),
    AgentDefinition(
        agent_id="analysis",
        name="Analysis Agent",
        description="Analyzes structured or unstructured data to identify patterns, trends, and insights. Best for data interpretation tasks.",
        handler=analysis_agent,
    ),
    AgentDefinition(
        agent_id="summarization",
        name="Summarization Agent",
        description="Condenses long documents or content into concise summaries. Best for distillation tasks.",
        handler=summarization_agent,
    ),
    AgentDefinition(
        agent_id="code",
        name="Code Agent",
        description="Generates, reviews, or explains code. Best for programming tasks.",
        handler=code_agent,
    ),
]

REGISTRY_MAP = {a.agent_id: a for a in AGENT_REGISTRY}

print("Agent registry initialized with agents:", [a.agent_id for a in AGENT_REGISTRY])

In [ ]:
import json


def build_orchestrator_prompt(user_request: str, registry: list[AgentDefinition]) -> str:
    agent_descriptions = "\n".join(
        f"  - {a.agent_id}: {a.description}" for a in registry
    )
    return f"""You are a workflow orchestrator. Given a user request, decide which agents to invoke and in what order.

Available agents:
{agent_descriptions}

User request: {user_request}

Respond with a JSON execution plan. Each step specifies:
  - agent_id: which agent to call
  - input: what to pass to the agent
  - depends_on: list of step indices this step waits for (empty = can run immediately)

Example response:
{{
  "reasoning": "This request needs research then analysis of the findings.",
  "steps": [
    {{"step": 0, "agent_id": "research", "input": "...", "depends_on": []}},
    {{"step": 1, "agent_id": "analysis", "input": "results from step 0", "depends_on": [0]}}
  ]
}}

Return only valid JSON."""


def simulate_orchestrator(user_request: str) -> dict:
    """
    Simulate orchestrator decision — replace with real LLM call in production.
    Maps request keywords to routing decisions for demo purposes.
    """
    request_lower = user_request.lower()

    if "code" in request_lower or "implement" in request_lower or "function" in request_lower:
        return {
            "reasoning": "This is a coding task. Route to code agent only.",
            "steps": [
                {"step": 0, "agent_id": "code", "input": user_request, "depends_on": []}
            ]
        }
    elif "summarize" in request_lower or "summary" in request_lower:
        return {
            "reasoning": "This is a summarization task.",
            "steps": [
                {"step": 0, "agent_id": "summarization", "input": user_request, "depends_on": []}
            ]
        }
    elif "analyze" in request_lower or "trends" in request_lower or "insights" in request_lower:
        return {
            "reasoning": "This needs research first, then analysis of the findings.",
            "steps": [
                {"step": 0, "agent_id": "research", "input": user_request, "depends_on": []},
                {"step": 1, "agent_id": "analysis", "input": "results from step 0", "depends_on": [0]},
                {"step": 2, "agent_id": "summarization", "input": "results from steps 0 and 1", "depends_on": [0, 1]},
            ]
        }
    else:
        return {
            "reasoning": "General research question.",
            "steps": [
                {"step": 0, "agent_id": "research", "input": user_request, "depends_on": []},
                {"step": 1, "agent_id": "summarization", "input": "results from step 0", "depends_on": [0]},
            ]
        }


print("Orchestrator routing logic ready")

In [ ]:
def execute_adaptive_workflow(user_request: str) -> dict:
    """
    Run the adaptive workflow:
    1. Orchestrator decides which agents to invoke and in what order
    2. Steps are executed in dependency order
    3. Each step receives the outputs of its dependencies as context
    4. Final outputs are returned
    """
    print("=" * 65)
    print(f"REQUEST: {user_request}")
    print("=" * 65)

    # Step 1: Orchestrator decides the plan
    print("\n[Orchestrator] Planning execution...")
    plan = simulate_orchestrator(user_request)
    print(f"[Orchestrator] Reasoning: {plan['reasoning']}")
    print(f"[Orchestrator] Steps: {[s['agent_id'] for s in plan['steps']]}")

    # Step 2: Execute steps in dependency order
    step_outputs: dict[int, str] = {}

    for step_def in plan["steps"]:
        step_idx = step_def["step"]
        agent_id = step_def["agent_id"]
        raw_input = step_def["input"]
        depends_on = step_def["depends_on"]

        # Build enriched input: append outputs from dependency steps
        if depends_on:
            dependency_context = "\n".join(
                f"Step {dep} output: {step_outputs[dep]}"
                for dep in depends_on
                if dep in step_outputs
            )
            enriched_input = f"{raw_input}\n\nContext from prior steps:\n{dependency_context}"
        else:
            enriched_input = raw_input

        # Execute the agent
        print(f"\n[Step {step_idx}] Invoking {agent_id}...")
        agent = REGISTRY_MAP.get(agent_id)
        if not agent:
            step_outputs[step_idx] = f"Error: unknown agent '{agent_id}'"
            continue

        output = agent.handler(enriched_input)
        step_outputs[step_idx] = output
        print(f"[Step {step_idx}] Output: {output}")

    # Step 3: Collect final output (last step's result, or synthesize)
    last_step = max(step_outputs.keys())
    final_output = step_outputs[last_step]

    print("\n" + "=" * 65)
    print("FINAL OUTPUT:")
    print(final_output)
    print("=" * 65)

    return {
        "request": user_request,
        "agents_invoked": [s["agent_id"] for s in plan["steps"]],
        "step_outputs": step_outputs,
        "final_output": final_output,
    }


# Demonstrate different routing paths for different request types
requests = [
    "Write a Python function to parse JSON from a file",
    "Analyze the latest trends in renewable energy adoption in Europe",
    "What is the current state of quantum computing?",
]

for req in requests:
    result = execute_adaptive_workflow(req)
    print(f"\nAgents used: {result['agents_invoked']}\n")

### Working Implementation with Real LLM Calls

The above uses a rule-based orchestrator for demonstration. In production, you replace `simulate_orchestrator` with an actual LLM call that reads the agent registry and produces a routing plan. The sub-agent handlers also call real LLMs.

**Setup**: set `OPENAI_API_KEY` in your environment before running.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from environment


def llm_orchestrate(user_request: str, registry: list[AgentDefinition]) -> dict:
    """
    Use an LLM to decide the execution plan from the agent registry.
    Returns a validated plan dict with 'reasoning' and 'steps'.
    """
    prompt = build_orchestrator_prompt(user_request, registry)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a workflow orchestrator. Always respond with valid JSON."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},  # enforce JSON output
    )

    plan = json.loads(response.choices[0].message.content)

    # Validate that the plan only references known agents
    known_ids = {a.agent_id for a in registry}
    for step in plan.get("steps", []):
        if step["agent_id"] not in known_ids:
            raise ValueError(f"Orchestrator referenced unknown agent: {step['agent_id']}")

    return plan


def llm_sub_agent(system_prompt: str, user_input: str, model: str = "gpt-4o") -> str:
    """Generic sub-agent executor using a system prompt for specialization."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input},
        ],
    )
    return response.choices[0].message.content


# Sub-agents backed by real LLM calls
REAL_AGENT_REGISTRY: list[AgentDefinition] = [
    AgentDefinition(
        agent_id="research",
        name="Research Agent",
        description="Finds and synthesizes factual information. Best for questions requiring up-to-date knowledge.",
        handler=lambda inp: llm_sub_agent(
            "You are a research specialist. Provide accurate, well-sourced information on the topic.",
            inp,
        ),
    ),
    AgentDefinition(
        agent_id="analysis",
        name="Analysis Agent",
        description="Analyzes data or research findings to extract patterns, trends, and insights.",
        handler=lambda inp: llm_sub_agent(
            "You are an expert analyst. Extract key trends, patterns, and actionable insights from the provided information.",
            inp,
        ),
    ),
    AgentDefinition(
        agent_id="summarization",
        name="Summarization Agent",
        description="Condenses content into concise, readable summaries.",
        handler=lambda inp: llm_sub_agent(
            "You are a summarization expert. Produce clear, concise summaries that capture the essential points.",
            inp,
        ),
    ),
    AgentDefinition(
        agent_id="code",
        name="Code Agent",
        description="Generates, reviews, and explains code.",
        handler=lambda inp: llm_sub_agent(
            "You are an expert software engineer. Write clean, well-documented code that follows best practices.",
            inp,
        ),
    ),
]


def execute_adaptive_workflow_real(user_request: str) -> dict:
    """Full adaptive workflow with real LLM orchestrator and sub-agents."""
    print(f"REQUEST: {user_request}\n")

    real_registry_map = {a.agent_id: a for a in REAL_AGENT_REGISTRY}

    # LLM decides the plan
    plan = llm_orchestrate(user_request, REAL_AGENT_REGISTRY)
    print(f"Orchestrator reasoning: {plan['reasoning']}")
    print(f"Plan: {[s['agent_id'] for s in plan['steps']]}\n")

    # Execute in dependency order
    step_outputs: dict[int, str] = {}
    for step_def in plan["steps"]:
        step_idx = step_def["step"]
        agent_id = step_def["agent_id"]
        depends_on = step_def.get("depends_on", [])

        enriched_input = step_def["input"]
        if depends_on:
            prior_context = "\n\n".join(
                f"Output from {plan['steps'][dep]['agent_id']}: {step_outputs[dep]}"
                for dep in depends_on
                if dep in step_outputs
            )
            enriched_input = f"{enriched_input}\n\n{prior_context}"

        print(f"Running {agent_id}...")
        agent = real_registry_map[agent_id]
        step_outputs[step_idx] = agent.handler(enriched_input)

    last_output = step_outputs[max(step_outputs.keys())]
    print(f"\nFinal output:\n{last_output}")
    return {"agents_invoked": [s["agent_id"] for s in plan["steps"]], "output": last_output}


# Requires OPENAI_API_KEY
result = execute_adaptive_workflow_real(
    "Analyze the key trends in large language model development in 2024 and 2025"
)
print(f"\nAgents invoked: {result['agents_invoked']}")

### Design Decisions

**How should the orchestrator communicate its decisions?**

JSON is the most practical format for production. Natural language routing instructions are hard to parse reliably. A structured JSON plan with explicit `agent_id`, `input`, and `depends_on` fields can be validated before execution — you can catch unknown agent references, circular dependencies, or malformed inputs before any sub-agent runs.

Using OpenAI's `response_format: json_object` or tool calling API makes this even more reliable than relying on prompt instructions alone.

**Sequential vs parallel execution**

The `depends_on` field in the plan naturally encodes parallelism. Steps with no dependencies can run simultaneously. Steps that depend on others must wait. A simple topological sort of the dependency graph gives you the correct execution order, and steps at the same level in the sorted graph can be parallelized.

```python
# Steps 0 and 1 have no dependencies — run in parallel
# Step 2 depends on both — runs after both complete
{"steps": [
    {"step": 0, "agent_id": "research", "depends_on": []},
    {"step": 1, "agent_id": "code", "depends_on": []},
    {"step": 2, "agent_id": "summarization", "depends_on": [0, 1]}
]}
```

**When should the orchestrator loop?**

The orchestrator can be called again after receiving sub-agent outputs, with a second routing decision: proceed to synthesis, retry a step, or escalate. This is the "adaptive iteration" aspect. A simple version just has a quality check step in the plan. A more sophisticated version uses a second orchestrator call after collecting outputs.

**How to prevent runaway loops**

Always set a maximum iteration count and surface the partial output to the user when it's hit, rather than failing silently. Log every routing decision with the request ID so you can reconstruct what happened.

### How It Fits with Other Patterns

| Pattern | Relationship to Adaptive Workflow |
|---------|----------------------------------|
| **Reflection** | Individual sub-agents can use reflection internally for quality improvement |
| **Tool Use** | Sub-agents are often tool-using agents; adaptive workflow coordinates between them |
| **ReAct** | The orchestrator itself can use ReAct as its reasoning loop for routing decisions |
| **Planning** | Planning produces a complete upfront plan; adaptive workflow makes routing decisions dynamically per step |
| **Multi-Agent** | Multi-agent has a fixed topology; adaptive workflow's topology changes per request |

Adaptive workflow is effectively a meta-pattern: it wraps and coordinates the other patterns. You'd typically find Reflection, Tool Use, and ReAct operating *inside* sub-agents in an Adaptive Workflow system, not as standalone alternatives to it.

The decision of when to use Planning vs Adaptive Workflow is worth dwelling on:
- **Planning**: "I know this task needs steps A, B, C — let me plan them all upfront then execute." Use when the task structure is predictable.
- **Adaptive Workflow**: "Different inputs need different agents — let me decide routing per-request." Use when request diversity is high and a fixed pipeline would be either wasteful or insufficient.

### Workflow Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                      USER REQUEST                               │
│      "Analyze trends in EV charging infrastructure in Europe"   │
└──────────────────────────┬──────────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                   ORCHESTRATOR (LLM)                            │
│                                                                 │
│  Reads agent registry, decides execution plan:                  │
│  Step 0: research ("EV charging Europe market data")            │
│  Step 1: analysis (depends on step 0)                           │
│  Step 2: summarization (depends on steps 0 and 1)               │
└──────┬──────────────────────────────────────────────────────────┘
       │
       ├──────────────────────────────────────────┐
       │                                          │
       ▼                                          │
┌──────────────────┐                              │
│  Research Agent  │  Step 0 (no dependencies     │
│                  │  — runs immediately)          │
│  Retrieves data  │                              │
│  on EV charging  │                              │
└────────┬─────────┘                              │
         │ output: market statistics,             │
         │ infrastructure gaps, key players       │
         ▼                                        │
┌──────────────────────────────────┐              │
│         Analysis Agent           │  Step 1      │
│  (depends on step 0 output)      │              │
│                                  │              │
│  Identifies 3 key growth trends, │              │
│  regional adoption differences,  │              │
│  infrastructure bottlenecks      │              │
└────────────────┬─────────────────┘              │
                 │                                │
                 ▼                                │
┌─────────────────────────────────────────────────┴────┐
│              Summarization Agent                      │  Step 2
│    (depends on steps 0 AND 1)                        │
│                                                       │
│    Condenses research + analysis into a               │
│    structured executive summary                       │
└──────────────────────────┬────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────────┐
│                    FINAL RESPONSE                               │
│   Comprehensive analysis with research data + insights          │
│   distilled into a concise, actionable summary                  │
└─────────────────────────────────────────────────────────────────┘

Compare: a simple "What is Python?" request routes only to the
research agent directly — no analysis or summarization needed.
The workflow topology adapts to the request.
```

### Summary

Adaptive Workflow is the most capable and most complex of the six patterns. It's the right choice when you need a system that can handle diverse request types efficiently — routing each to the right combination of agents — rather than forcing every request through the same fixed pipeline.

The core components are: an LLM orchestrator, a structured agent registry, sub-agents with well-defined responsibilities, and a synthesis step. The orchestrator's routing plan is the critical element — it needs to be validated before execution, logged for debugging, and bounded to prevent runaway loops.

Key decisions when building it:
- How does the orchestrator communicate its plan? (JSON with explicit dependency graph is most reliable.)
- Are sub-agent steps sequential or parallelizable? (Use a dependency graph to encode this.)
- Does the orchestrator get a second pass after collecting outputs, or is routing one-shot? (Second pass enables adaptive iteration, but adds latency and complexity.)
- What is the escalation path when quality checks fail?

Start simple: a one-shot orchestrator, sequential execution, and a separate synthesizer. Add parallelism and iterative routing once the basics are stable.

### Papers

- **AutoGen** (Wu et al., 2023) — multi-agent conversation framework with dynamic agent coordination
- **CrewAI** — role-based agent framework with configurable task routing
- **LangGraph** — graph-based orchestration of LLM workflows with stateful routing
- **HuggingGPT** (Shen et al., 2023) — using an LLM as a controller to assign tasks to specialized models